In [1]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_country_master = (
    spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load("abfss://Dell_SEO_Analytics@onelake.dfs.fabric.microsoft.com/LH_SEO_Bronze.Lakehouse/Files/bronze/country_master/country_master.csv")
)

print("Rows:", df_country_master.count())
print("Columns:", len(df_country_master.columns))

display(df_country_master)


StatementMeta(, c985b49a-8725-412f-89cc-e2ac7f3ff605, 3, Finished, Available, Finished, False)

Rows: 10
Columns: 6


SynapseWidget(Synapse.DataFrame, 677e1be0-9c9a-4568-9c21-1034fdb6b395)

In [2]:
print("Duplicate country codes:")

duplicate_country_codes = (
    df_country_master
    .groupBy("country_code")
    .count()
    .filter(F.col("count") > 1)
)

print(duplicate_country_codes.count())

StatementMeta(, c985b49a-8725-412f-89cc-e2ac7f3ff605, 4, Finished, Available, Finished, False)

Duplicate country codes:
0


In [3]:
display(
    df_country_master
    .select(
        "country_code",
        "country_name",
        "region",
        "sub_region",
        "active_flag"
    )
    .orderBy("country_code")
)

StatementMeta(, c985b49a-8725-412f-89cc-e2ac7f3ff605, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ee08f5cf-6f8e-402b-836a-10d3b5cf79f6)

In [4]:
print("Null counts:")

for column_name in [
    "country_code",
    "country_name",
    "region",
    "sub_region"
]:
    null_count = (
        df_country_master
        .filter(F.col(column_name).isNull())
        .count()
    )

    print(f"{column_name}: {null_count}")

StatementMeta(, c985b49a-8725-412f-89cc-e2ac7f3ff605, 6, Finished, Available, Finished, False)

Null counts:
country_code: 0
country_name: 0
region: 0
sub_region: 0


In [5]:
print("Null counts:")

for column_name in [
    "country_code",
    "country_name",
    "region",
    "sub_region"
]:
    null_count = (
        df_country_master
        .filter(F.col(column_name).isNull())
        .count()
    )

    print(f"{column_name}: {null_count}")

StatementMeta(, c985b49a-8725-412f-89cc-e2ac7f3ff605, 7, Finished, Available, Finished, False)

Null counts:
country_code: 0
country_name: 0
region: 0
sub_region: 0


In [6]:
df_country_dim = (
    df_country_master
    .withColumn(
        "country_code",
        F.upper(F.trim(F.col("country_code")))
    )
    .withColumn(
        "country_name",
        F.trim(F.col("country_name"))
    )
    .withColumn(
        "region",
        F.trim(F.col("region"))
    )
    .withColumn(
        "sub_region",
        F.trim(F.col("sub_region"))
    )
    .withColumn(
        "active_flag",
        F.trim(F.col("active_flag"))
    )
)

StatementMeta(, c985b49a-8725-412f-89cc-e2ac7f3ff605, 8, Finished, Available, Finished, False)

In [7]:
df_country_dim = (
    df_country_master
    .withColumn(
        "country_code",
        F.upper(F.trim(F.col("country_code")))
    )
    .withColumn(
        "country_name",
        F.trim(F.col("country_name"))
    )
    .withColumn(
        "region",
        F.trim(F.col("region"))
    )
    .withColumn(
        "sub_region",
        F.trim(F.col("sub_region"))
    )
    .withColumn(
        "active_flag",
        F.trim(F.col("active_flag"))
    )
)

StatementMeta(, c985b49a-8725-412f-89cc-e2ac7f3ff605, 9, Finished, Available, Finished, False)

In [8]:
display(
    df_country_dim
    .select(
        "country_code",
        "country_name",
        "region",
        "sub_region",
        "active_flag"
    )
    .orderBy("country_code")
)

StatementMeta(, c985b49a-8725-412f-89cc-e2ac7f3ff605, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 752e8385-4e16-431e-8621-7bc68ca2d679)

In [9]:
(
    df_country_dim.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("dim_country")
)

print("Silver dim_country table created successfully.")

StatementMeta(, c985b49a-8725-412f-89cc-e2ac7f3ff605, 11, Finished, Available, Finished, False)

Silver dim_country table created successfully.


In [10]:
df_silver_country = spark.read.table("dim_country")

print(
    "Silver dim_country rows:",
    df_silver_country.count()
)

print(
    "Silver dim_country columns:",
    len(df_silver_country.columns)
)

display(df_silver_country)

StatementMeta(, c985b49a-8725-412f-89cc-e2ac7f3ff605, 12, Finished, Available, Finished, False)

Silver dim_country rows: 10
Silver dim_country columns: 6


SynapseWidget(Synapse.DataFrame, c30b31b6-0314-462b-86c2-cb7a3517ddb5)